Brute Force Algorithm

In [ ]:
import math

def brute_force_closest_points(P):
    n = len(P)
    d_min = float('inf')
    index_1 = -1
    index_2 = -1

    for i in range(n - 1):  # i from 0 to n-2
        for j in range(i + 1, n):  # j from i+1 to n-1
            d = math.sqrt((P[i][0] - P[j][0])**2 + (P[i][1] - P[j][1])**2)
            if d < d_min:
                d_min = d
                index_1 = i
                index_2 = j

    return index_1 + 1, index_2 + 1, d_min  # Return as 1-based index

def main():
    n = int(input("Enter the number of points (n ≥ 2): "))
    if n < 2:
        print("At least 2 points are required.")
        return

    points = []
    for i in range(n):
        x, y = map(int, input(f"Enter coordinates for point {i+1} (format: x y): ").split())
        points.append((x, y))

    index1, index2, min_dist = brute_force_closest_points(points)
    print(f"\nClosest pair: P{index1} and P{index2}")
    print(f"Distance: {min_dist:.4f}")

if __name__ == "__main__":
    main()





Enter the number of points (n ≥ 2): 3
Enter coordinates for point 1 (format: x y): 10  12
Enter coordinates for point 2 (format: x y): 4  7
Enter coordinates for point 3 (format: x y): 9  6

Closest pair: P2 and P3
Distance: 5.0990


both

In [ ]:
import random
import time
import math
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional

# Type alias for a point
Point = Tuple[float, float]

def generate_unique_points(n: int) -> List[Point]:
    """Generate n unique random points with x,y coordinates between 0 and 10000."""
    points = set()
    while len(points) < n:
        x = random.uniform(0, 10000)
        y = random.uniform(0, 10000)
        points.add((x, y))
    return list(points)

def distance(p1: Point, p2: Point) -> float:
    """Calculate Euclidean distance between two points."""
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def brute_force(points: List[Point]) -> Tuple[float, Optional[Point], Optional[Point]]:
    """
    Find the closest pair of points using brute force approach.
    Returns (min_distance, point1, point2)
    """
    n = len(points)
    if n < 2:
        return float('inf'), None, None

    min_dist = float('inf')
    closest_pair = (None, None)

    for i in range(n):
        for j in range(i + 1, n):
            dist = distance(points[i], points[j])
            if dist < min_dist:
                min_dist = dist
                closest_pair = (points[i], points[j])

    return min_dist, closest_pair[0], closest_pair[1]

def closest_pair_strip(strip: List[Point], d: float) -> Tuple[float, Optional[Point], Optional[Point]]:
    """
    Find the closest pair of points in a strip of width 2d.
    Returns (min_distance, point1, point2)
    """
    min_dist = d
    closest_pair = (None, None)

    # Sort points by y-coordinate
    strip.sort(key=lambda point: point[1])

    # Compare each point with at most 7 points ahead (proven optimal)
    for i in range(len(strip)):
        j = i + 1
        while j < len(strip) and strip[j][1] - strip[i][1] < min_dist:
            dist = distance(strip[i], strip[j])
            if dist < min_dist:
                min_dist = dist
                closest_pair = (strip[i], strip[j])
            j += 1

    return min_dist, closest_pair[0], closest_pair[1]

def closest_pair_recursive(points_x: List[Point], points_y: List[Point]) -> Tuple[float, Optional[Point], Optional[Point]]:
    """
    Recursive function to find closest pair of points using divide and conquer.
    points_x: Points sorted by x-coordinate
    points_y: Points sorted by y-coordinate
    Returns (min_distance, point1, point2)
    """
    n = len(points_x)

    # Base case: if there are <= 3 points, use brute force
    if n <= 3:
        return brute_force(points_x)

    # Divide the points into left and right halves
    mid = n // 2
    mid_point = points_x[mid]

    # Divide points into left and right halves (preserving their order)
    points_x_left = points_x[:mid]
    points_x_right = points_x[mid:]

    # Split points_y into left and right halves based on the x-coordinate of mid_point
    points_y_left = []
    points_y_right = []
    for point in points_y:
        if point[0] <= mid_point[0]:
            points_y_left.append(point)
        else:
            points_y_right.append(point)

    # Recursively find closest pairs in left and right halves
    d_left, p1_left, p2_left = closest_pair_recursive(points_x_left, points_y_left)
    d_right, p1_right, p2_right = closest_pair_recursive(points_x_right, points_y_right)

    # Find the smaller of the two distances
    if d_left < d_right:
        d = d_left
        best_pair = (p1_left, p2_left)
    else:
        d = d_right
        best_pair = (p1_right, p2_right)

    # Build a strip of points within distance d of the middle line
    strip = []
    for point in points_y:
        if abs(point[0] - mid_point[0]) < d:
            strip.append(point)

    # Find the closest pair in the strip (if any)
    strip_dist, p1_strip, p2_strip = closest_pair_strip(strip, d)

    # Return the minimum of d and strip_dist
    if strip_dist < d:
        return strip_dist, p1_strip, p2_strip
    else:
        return d, best_pair[0], best_pair[1]

def divide_and_conquer(points: List[Point]) -> Tuple[float, Optional[Point], Optional[Point]]:
    """
    Find the closest pair of points using divide and conquer approach.
    Returns (min_distance, point1, point2)
    """
    if len(points) < 2:
        return float('inf'), None, None

    # Sort points by x and y coordinates
    points_x = sorted(points, key=lambda point: point[0])
    points_y = sorted(points, key=lambda point: point[1])

    return closest_pair_recursive(points_x, points_y)

def run_benchmark():
    # Input sizes to test
    input_sizes = [10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000]
    iterations = 10

    # To store results
    brute_force_times = []
    divide_conquer_times = []

    for n in input_sizes:
        print(f"\nTesting with {n} points:")
        bf_total = 0
        dc_total = 0

        for i in range(iterations):
            # Generate the same set of points for both algorithms
            points = generate_unique_points(n)

            # Brute Force
            start_time = time.time()
            bf_result = brute_force(points)
            end_time = time.time()
            bf_time = end_time - start_time
            bf_total += bf_time

            # Divide and Conquer
            start_time = time.time()
            dc_result = divide_and_conquer(points)
            end_time = time.time()
            dc_time = end_time - start_time
            dc_total += dc_time

            print(f"  Iteration {i+1}: Brute Force: {bf_time:.4f}s, Divide & Conquer: {dc_time:.4f}s")

            # Verify results match
            if abs(bf_result[0] - dc_result[0]) > 1e-9:
                print(f"  WARNING: Results differ! BF: {bf_result[0]}, D&C: {dc_result[0]}")

        # Calculate average times
        avg_bf = bf_total / iterations
        avg_dc = dc_total / iterations

        brute_force_times.append(avg_bf)
        divide_conquer_times.append(avg_dc)

        print(f"Average for {n} points: Brute Force: {avg_bf:.4f}s, Divide & Conquer: {avg_dc:.4f}s")

    # Plot the results
    plt.figure(figsize=(10, 6))
    plt.plot(input_sizes, brute_force_times, 'o-', label='Brute Force')
    plt.plot(input_sizes, divide_conquer_times, 's-', label='Divide & Conquer')
    plt.xlabel('Number of Points')
    plt.ylabel('Running Time (seconds)')
    plt.title('Algorithm Performance Comparison')
    plt.legend()
    plt.grid(True)
    plt.savefig('closest_pair_performance.png')
    plt.show()

    # Print summary
    print("\nPerformance Summary:")
    print(f"{'Input Size':<12} {'Brute Force':<15} {'Divide & Conquer':<15} {'Ratio (BF/DC)':<15}")
    print("-" * 60)
    for i, n in enumerate(input_sizes):
        ratio = brute_force_times[i] / divide_conquer_times[i] if divide_conquer_times[i] > 0 else float('inf')
        print(f"{n:<12} {brute_force_times[i]:<15.4f} {divide_conquer_times[i]:<15.4f} {ratio:<15.2f}")

if __name__ == "__main__":
    # Run a small example to verify correctness
    test_points = [(0, 0), (1, 1), (2, 2), (3, 3), (0.5, 0.5)]
    bf_dist, bf_p1, bf_p2 = brute_force(test_points)
    dc_dist, dc_p1, dc_p2 = divide_and_conquer(test_points)

    print("Small test case:")
    print(f"Brute force: {bf_dist:.4f} between {bf_p1} and {bf_p2}")
    print(f"Divide & Conquer: {dc_dist:.4f} between {dc_p1} and {dc_p2}")

    # Run the benchmark
    print("\nStarting benchmark...")
    run_benchmark()

Small test case:
Brute force: 0.7071 between (0, 0) and (0.5, 0.5)
Divide & Conquer: 0.7071 between (0, 0) and (0.5, 0.5)

Starting benchmark...

Testing with 10000 points:
  Iteration 1: Brute Force: 20.4580s, Divide & Conquer: 0.0459s
  Iteration 2: Brute Force: 20.0103s, Divide & Conquer: 0.0785s
  Iteration 3: Brute Force: 19.8721s, Divide & Conquer: 0.0456s
  Iteration 4: Brute Force: 20.3643s, Divide & Conquer: 0.0459s
  Iteration 5: Brute Force: 19.2567s, Divide & Conquer: 0.0516s
  Iteration 6: Brute Force: 20.3497s, Divide & Conquer: 0.0473s
  Iteration 7: Brute Force: 19.9309s, Divide & Conquer: 0.0829s
  Iteration 8: Brute Force: 20.0542s, Divide & Conquer: 0.0456s
  Iteration 9: Brute Force: 20.6145s, Divide & Conquer: 0.0470s
  Iteration 10: Brute Force: 19.3767s, Divide & Conquer: 0.0540s
Average for 10000 points: Brute Force: 20.0287s, Divide & Conquer: 0.0544s

Testing with 20000 points:
  Iteration 1: Brute Force: 106.1854s, Divide & Conquer: 0.1092s
  Iteration 2: Bru